# Interactive COC Merger Rates Figures

This notebook creates interactive HTML versions of the COC merger rate figures using Plotly.
Hover over scatter points to see detailed information about each model.

## 1 — Imports

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from pathlib import Path

## 2 — Load data

In [ ]:
# Load isolated binary evolution rates
data_dir = Path('/Users/floorbroekgaarden/Projects/GitHub/Rates_of_Compact_Object_Coalescence/Data_Mandel_and_Broekgaarden_2026')
rates_csv = data_dir / 'isolated-binary-evolution.csv'
relationships_csv = data_dir / 'isolated-binary-evolution_relationships.csv'

# Load rates data
rates_df = pd.read_csv(rates_csv, dtype={"year": "Int64", "month": "Int64"})
rates_df = rates_df[rates_df['formation_channel'] == 'isolated-binary-evolution']

# Load relationships data
relationships_df = pd.read_csv(relationships_csv)

print(f"Loaded {len(rates_df)} rate entries")
print(f"Loaded {len(relationships_df)} relationships")
print(f"\\nDCO types: {rates_df['compact_object_type'].unique()}")

## 3 — Create hover text helper

In [ ]:
def create_hover_text(row, relation_row=None):
    """
    Create detailed hover text for a data point.
    Shows: study, arxiv link, submodel, parameter info, and rate.
    """
    text = f"<b>{row['label']}</b><br>"
    text += f"Study: {row['study_key']}<br>"
    
    # Add arxiv link if available
    if pd.notna(row.get('arxiv_url')) and row['arxiv_url'] != '':
        text += f"arXiv: <a href='{row['arxiv_url']}' target='_blank'>link</a><br>"
    
    # Add model information
    if pd.notna(row.get('submodel string')) and row['submodel string'] != '':
        text += f"Submodel: {row['submodel string']}<br>"
    
    # Add relationship information if available
    if relation_row is not None:
        text += f"Parameter Family: {relation_row['parameter_family']}<br>"
        text += f"Parameter: {relation_row['parameter']}<br>"
        text += f"From Model: {relation_row['from_submodel']}<br>"
        text += f"To Model: {relation_row['to_submodel']}<br>"
    
    # Add rate
    rate = row.get('rate_Gpc3yr')
    if pd.notna(rate):
        text += f"Rate: {rate:.1f} Gpc<sup>-3</sup> yr<sup>-1</sup>"
    
    return text

## 4 — Create interactive scatter plots by DCO type

In [ ]:
def create_interactive_figure(dco_type, rates_data, relationships_data):
    """
    Create an interactive scatter plot for a given DCO type.
    X-axis: publication year, Y-axis: merger rate (log scale)
    Hover to see: study, submodel, parameter info, rate, arxiv link
    """
    # Filter data for this DCO type
    dco_data = rates_data[rates_data['compact_object_type'] == dco_type].copy()
    dco_data = dco_data.dropna(subset=['rate_Gpc3yr'])
    dco_data = dco_data[dco_data['rate_Gpc3yr'] > 0]
    
    # Create figure
    fig = go.Figure()
    
    # Group by study for coloring and markers
    for study in sorted(dco_data['study_key'].unique()):
        study_data = dco_data[dco_data['study_key'] == study].copy()
        
        # Create hover text for each point
        hover_texts = []
        for idx, row in study_data.iterrows():
            # Try to find relationship info for this model
            rel_data = relationships_data[
                (relationships_data['study_key'] == study) & 
                ((relationships_data['to_submodel'] == row.get('submodel')) |
                 (relationships_data['from_submodel'] == row.get('submodel')))
            ]
            rel_row = rel_data.iloc[0] if len(rel_data) > 0 else None
            hover_texts.append(create_hover_text(row, rel_row))
        
        fig.add_trace(go.Scatter(
            x=study_data['year'],
            y=study_data['rate_Gpc3yr'],
            mode='markers',
            name=study,
            hovertext=hover_texts,
            hoverinfo='text',
            marker=dict(
                size=9,
                opacity=0.7,
                line=dict(width=1.5, color='white')
            )
        ))
    
    # Update layout
    fig.update_layout(
        title=f"<b>{dco_type} Merger Rates — Interactive</b>",
        xaxis_title="Publication Year",
        yaxis_title="Merger Rate [Gpc⁻³ yr⁻¹]",
        hovermode='closest',
        height=700,
        width=1200,
        template='plotly_white',
        yaxis_type='log',
        xaxis=dict(gridwidth=1, gridcolor='lightgray'),
        yaxis=dict(gridwidth=1, gridcolor='lightgray'),
        font=dict(size=12),
        legend=dict(x=1.02, y=1, xanchor='left', yanchor='top')
    )
    
    return fig

# Create figures for each DCO type
dco_types = ['BH-BH', 'BH-NS', 'NS-NS']
figures = {}

for dco_type in dco_types:
    fig = create_interactive_figure(dco_type, rates_df, relationships_df)
    figures[dco_type] = fig
    print(f"✓ Created interactive figure for {dco_type}")

## 5 — Save as HTML

In [ ]:
# Create output directory
output_dir = data_dir / 'interactive_plots'
output_dir.mkdir(exist_ok=True)

# Save figures as standalone HTML files
for dco_type, fig in figures.items():
    output_file = output_dir / f"merger_rates_{dco_type.replace('-', '_')}.html"
    fig.write_html(
        str(output_file),
        config={'responsive': True, 'displayModeBar': True}
    )
    print(f"  Saved: {output_file}")

print(f"\n✓ All interactive plots saved to {output_dir}")

## 6 — Display figures (in Jupyter)

In [ ]:
# Display in notebook for preview
for dco_type, fig in figures.items():
    print(f"\n### {dco_type}")
    fig.show()